# Module 1: YOLOv8 物件偵測 — 讓 AI 看懂世界

## 學習目標
- 理解 YOLO 物件偵測的概念
- 對即時串流做物件偵測
- 體驗 6 種不同場景的 AI 辨識

## 什麼是 YOLO？

**YOLO = You Only Look Once**（你只需要看一次）

傳統方法是用滑動視窗，一小塊一小塊掃描整張圖。
YOLO 直接「看」整張圖，一次找出所有物件。

```
傳統方法（慢）          YOLO（快）
┌──┐                   ┌─────────────┐
│掃│→ 掃 → 掃 → ...    │ 一次看整張圖 │
│描│  一小塊一小塊       │ 同時找出所有 │
│  │  非常耗時          │ 物件的位置   │
└──┘                   └─────────────┘
```

> 先跑出結果，再回頭理解原理！

---
## Step 1: 載入環境（如果從新的 Runtime 開始，先執行這裡）

In [ ]:
# === 環境設定（每次開新 Runtime 都要跑一次）===
!pip install ultralytics opencv-python-headless pillow easyocr -q

import cv2
import numpy as np
from PIL import Image
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
from collections import Counter

plt.rcParams['font.size'] = 12

def show_frame(frame, title='', size=(640, 360)):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb).resize(size)
    if title:
        print(title)
    display(img)

def grab_frame(url, timeout=10):
    cap = cv2.VideoCapture(url)
    cap.set(cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, timeout * 1000)
    ret, frame = cap.read()
    cap.release()
    if ret:
        return frame
    raise ConnectionError(f'串流連線失敗，請換一個來源')

# 串流來源
STREAMS = {
    'Al Jazeera 半島電視台': 'https://live-hls-apps-aje-fa.getaj.net/AJE/index.m3u8',
    'DW 德國之聲': 'https://dwamdstream102.akamaized.net/hls/live/2015525/dwstream102/master.m3u8',
    'France 24 法國': 'https://amg00106-france24-france24-samsunguk-qvpp8.amagi.tv/playlist/amg00106-france24-france24-samsunguk/playlist.m3u8',
    'NHK World 日本': 'https://nhkwlive-xjp.akamaized.net/hls/live/2003458/nhkwlive-xjp/index_1080p.m3u8',
    'CGTN 中國': 'https://amg00405-rakutentv-cgtn-rakuten-i9tar.amagi.tv/master.m3u8',
    'ABC News 美國': 'https://abc-news-dmd-streams-1.akamaized.net/out/v1/701126012d044971b3fa89406a440133/index.m3u8',
    'QVC 購物台': 'https://qvc-amd-live.akamaized.net/hls/live/2034113/lsqvc2us/master.m3u8',
    'HSN 家庭購物': 'https://a-cdn.klowdtv.com/live2/hsn_720p/playlist.m3u8',
    'QVC UK 英國': 'https://qvcuk-live.akamaized.net/hls/live/2097112/qvc/master.m3u8',
    'Shop Channel 日本': 'https://stream3.shopch.jp/HLS/master.m3u8',
    'WildEarth 野生地球': 'https://dqga3jatxofgx.cloudfront.net/WildEarth.m3u8',
    'Tanzania Safari 坦尚尼亞': 'https://stream-134630.castr.net/5fe35eae8c53540cab83659a/live_31dabe40323511f08b8efff0016f3b67/index.m3u8',
    'BBC Earth 亞洲': 'https://cdn4.skygo.mn/live/disk1/BBC_earth/HLSv3-FTA/BBC_earth.m3u8',
    'Red Bull TV': 'https://3ea22335.wurl.com/master/f36d25e7e52f1ba8d7e56eb859c636563214f541/UmFrdXRlblRWLWdiX1JlZEJ1bGxUVl9ITFM/playlist.m3u8',
    'beIN SPORTS': 'https://d35j504z0x2vu2.cloudfront.net/v1/master/0bc8e8376bd8417a1b6761138aa41c26c7309312/bein-sports-xtra/playlist.m3u8',
    '4K Travel TV': 'https://d35j504z0x2vu2.cloudfront.net/v1/master/0bc8e8376bd8417a1b6761138aa41c26c7309312/4k-travel-tv/manifest.m3u8',
    'DroneTV 空拍': 'https://d35j504z0x2vu2.cloudfront.net/v1/master/0bc8e8376bd8417a1b6761138aa41c26c7309312/dronetv/playlist.m3u8',
}

print('環境載入完成！')

---
## Step 2: 載入 YOLO 模型

In [ ]:
from ultralytics import YOLO

# 載入預訓練模型（第一次會自動下載，約 6MB）
# nano 版速度最快，適合即時偵測
model = YOLO('yolov8n.pt')

print('YOLO 模型載入完成！')
print(f'此模型可以辨識 {len(model.names)} 種物件：')
print()

In [ ]:
# === YOLO 能辨識的 80 種物件一覽 ===
# 分類整理，方便查看

categories = {
    '人物': [],
    '交通工具': [],
    '動物': [],
    '運動用品': [],
    '食物': [],
    '家具家電': [],
    '日常用品': [],
}

cat_map = {
    0: '人物',
    **{i: '交通工具' for i in [1,2,3,4,5,6,7,8]},
    **{i: '動物' for i in [14,15,16,17,18,19,20,21,22,23]},
    **{i: '運動用品' for i in [24,25,26,27,28,29,30,31,32,33,34,35]},
    **{i: '食物' for i in [46,47,48,49,50,51,52,53,54,55]},
    **{i: '家具家電' for i in [56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72]},
}

for idx, name in model.names.items():
    cat = cat_map.get(idx, '日常用品')
    categories[cat].append(name)

for cat, items in categories.items():
    if items:
        print(f'{cat} ({len(items)}種): {" | ".join(items)}')
        print()

---
## Step 3: 第一次 AI 物件偵測！

讓 YOLO 來看一幀即時串流畫面，看它能找到什麼。

In [ ]:
# === 你的第一次 AI 偵測 ===
channel = 'Al Jazeera 半島電視台'  # 可換成其他頻道

# 1. 抓取即時畫面
frame = grab_frame(STREAMS[channel])

# 2. 用 YOLO 偵測
results = model(frame)

# 3. 畫出偵測結果（自動標框 + 標籤）
annotated = results[0].plot()

# 4. 顯示原圖 vs 偵測結果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('YOLO Detection')
axes[1].axis('off')
plt.suptitle(f'{channel}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === 看看 YOLO 偵測到了什麼 ===
print(f'偵測結果：共找到 {len(results[0].boxes)} 個物件')
print('=' * 50)

for i, box in enumerate(results[0].boxes):
    cls_name = model.names[int(box.cls)]      # 物件類別
    conf = float(box.conf)                     # 信心度 (0~1)
    x1, y1, x2, y2 = box.xyxy[0].tolist()     # 框的座標
    w = x2 - x1
    h = y2 - y1
    print(f'  [{i+1}] {cls_name:15s}  信心度: {conf:.1%}  位置: ({x1:.0f},{y1:.0f})  大小: {w:.0f}x{h:.0f}')

# 統計各類別數量
counts = Counter(model.names[int(box.cls)] for box in results[0].boxes)
print()
print('物件統計:')
for cls, cnt in counts.most_common():
    print(f'  {cls}: {cnt} 個')

### YOLO 偵測結果圖解

每個偵測結果包含 3 個資訊：

```
┌─────────────────────────────┐
│  person 0.95                │ ← 類別名稱 + 信心度
│  ┌───────────┐              │
│  │           │              │
│  │  偵測到的  │ ← Bounding Box（邊界框）
│  │   物件     │    = (x1, y1, x2, y2)
│  │           │
│  └───────────┘              │
│                             │
└─────────────────────────────┘

信心度 (Confidence):
  0.95 = AI 有 95% 把握這是一個人
  0.50 = AI 只有 50% 把握（不太確定）
  
信心度越高 → 偵測越可靠
```

In [ ]:
# === 圖解：信心度門檻 (Confidence Threshold) ===
# 調整門檻值，看看偵測結果有什麼不同

thresholds = [0.25, 0.50, 0.75]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, thresh in zip(axes, thresholds):
    r = model(frame, conf=thresh, verbose=False)
    annotated = r[0].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    n = len(r[0].boxes)
    ax.set_title(f'Threshold={thresh}  ({n} objects)')
    ax.axis('off')

plt.suptitle('Confidence Threshold Effect', fontsize=14)
plt.tight_layout()
plt.show()

print('門檻越低 → 偵測到越多物件，但可能包含誤判')
print('門檻越高 → 只保留高信心度的結果，更精確但可能漏掉')

---
## Step 4: 六大場景實戰

現在來用不同類型的串流，體驗 AI 在各種場景的表現！

### 場景 A: 新聞台 — 人物偵測
新聞畫面通常有主播、受訪者、現場記者，適合練習人物偵測。

In [ ]:
# === 場景 A: 新聞台人物偵測 ===
news_channels = ['Al Jazeera 半島電視台', 'DW 德國之聲', 'France 24 法國', 'NHK World 日本']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, ch in zip(axes.flat, news_channels):
    try:
        f = grab_frame(STREAMS[ch])
        r = model(f, classes=[0], verbose=False)  # classes=[0] = 只偵測人
        annotated = r[0].plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        n_people = len(r[0].boxes)
        ax.set_title(f'{ch} ({n_people} people)')
    except Exception as e:
        ax.text(0.5, 0.5, f'{ch}\nOffline', ha='center', va='center', fontsize=14)
    ax.axis('off')

plt.suptitle('Scene A: News - Person Detection', fontsize=16)
plt.tight_layout()
plt.show()

### 場景 B: 購物台 — 日常物件辨識
購物台畫面充滿各種商品，是測試物件辨識的好素材！

In [ ]:
# === 場景 B: 購物台物件偵測 ===
shop_channels = ['QVC 購物台', 'HSN 家庭購物', 'QVC UK 英國', 'Shop Channel 日本']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, ch in zip(axes.flat, shop_channels):
    try:
        f = grab_frame(STREAMS[ch])
        r = model(f, verbose=False)
        annotated = r[0].plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        counts = Counter(model.names[int(b.cls)] for b in r[0].boxes)
        top3 = ', '.join(f'{k}:{v}' for k, v in counts.most_common(3))
        ax.set_title(f'{ch}\n{top3}')
    except Exception as e:
        ax.text(0.5, 0.5, f'{ch}\nOffline', ha='center', va='center', fontsize=14)
    ax.axis('off')

plt.suptitle('Scene B: Shopping - Object Detection', fontsize=16)
plt.tight_layout()
plt.show()

### 場景 C: 野生動物頻道 — 動物辨識

YOLO 可以辨識的動物：bird, cat, dog, horse, sheep, cow, elephant, bear, zebra, giraffe

In [ ]:
# === 場景 C: 野生動物辨識 ===
# YOLO 的動物類別編號
animal_classes = [14, 15, 16, 17, 18, 19, 20, 21, 22, 23]

wildlife_channels = ['WildEarth 野生地球', 'Tanzania Safari 坦尚尼亞', 'BBC Earth 亞洲']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, ch in zip(axes, wildlife_channels):
    try:
        f = grab_frame(STREAMS[ch])
        # 偵測所有物件，但特別標注動物
        r = model(f, verbose=False)
        annotated = r[0].plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        animals = [model.names[int(b.cls)] for b in r[0].boxes if int(b.cls) in animal_classes]
        if animals:
            ax.set_title(f'{ch}\nAnimals: {", ".join(animals)}')
        else:
            ax.set_title(f'{ch}\n(No animals detected)')
    except Exception as e:
        ax.text(0.5, 0.5, f'{ch}\nOffline', ha='center', va='center', fontsize=14)
    ax.axis('off')

plt.suptitle('Scene C: Wildlife - Animal Detection', fontsize=16)
plt.tight_layout()
plt.show()

print('提示：野生動物頻道的內容隨時間變化，有時是紀錄片畫面，有時是實況攝影機')
print('如果沒偵測到動物，試著多抓幾次（重新執行這個 Cell）！')

### 場景 D: 運動賽事 — 快速移動物體追蹤

運動畫面是最有挑戰性的場景：物體移動快、畫面切換頻繁。

In [ ]:
# === 場景 D: 運動賽事偵測 ===
sports_channels = ['Red Bull TV', 'beIN SPORTS']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ch in zip(axes, sports_channels):
    try:
        f = grab_frame(STREAMS[ch])
        r = model(f, verbose=False)
        annotated = r[0].plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        counts = Counter(model.names[int(b.cls)] for b in r[0].boxes)
        summary = ', '.join(f'{k}:{v}' for k, v in counts.most_common(5))
        ax.set_title(f'{ch}\n{summary}')
    except Exception as e:
        ax.text(0.5, 0.5, f'{ch}\nOffline', ha='center', va='center', fontsize=14)
    ax.axis('off')

plt.suptitle('Scene D: Sports - Fast Object Detection', fontsize=16)
plt.tight_layout()
plt.show()

### 場景 E: 旅遊/街景 — 交通工具與行人

旅遊頻道和空拍畫面有大量的車輛、行人、建築，適合模擬交通監控。

YOLO 可偵測的交通工具：bicycle, car, motorcycle, airplane, bus, train, truck, boat

In [ ]:
# === 場景 E: 街景 / 交通偵測 ===
vehicle_classes = [1, 2, 3, 4, 5, 6, 7, 8]  # 交通工具類別

travel_channels = ['4K Travel TV', 'DroneTV 空拍']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ch in zip(axes, travel_channels):
    try:
        f = grab_frame(STREAMS[ch])
        r = model(f, verbose=False)
        annotated = r[0].plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        vehicles = [model.names[int(b.cls)] for b in r[0].boxes if int(b.cls) in vehicle_classes]
        people = sum(1 for b in r[0].boxes if int(b.cls) == 0)
        ax.set_title(f'{ch}\nVehicles: {len(vehicles)} | People: {people}')
    except Exception as e:
        ax.text(0.5, 0.5, f'{ch}\nOffline', ha='center', va='center', fontsize=14)
    ax.axis('off')

plt.suptitle('Scene E: Travel/Street - Traffic Detection', fontsize=16)
plt.tight_layout()
plt.show()

### 場景 F: 自選頻道 — 你來試試！

從串流清單中挑一個你感興趣的頻道，看 YOLO 能偵測到什麼。

In [ ]:
# === 動手做！選一個頻道 ===
# 可用的頻道名稱:
for name in STREAMS:
    print(f'  - {name}')

In [ ]:
# === 改這裡！填入你選的頻道 ===
my_channel = 'NHK World 日本'

frame = grab_frame(STREAMS[my_channel])
results = model(frame, verbose=False)
annotated = results[0].plot()

# 顯示結果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Detection ({len(results[0].boxes)} objects)')
axes[1].axis('off')
plt.suptitle(f'Your Choice: {my_channel}', fontsize=14)
plt.tight_layout()
plt.show()

# 印出統計
counts = Counter(model.names[int(b.cls)] for b in results[0].boxes)
print('\n偵測統計:')
for cls, cnt in counts.most_common():
    bar = '█' * cnt
    print(f'  {cls:15s} {bar} ({cnt})')

---
## Step 5: 即時連續偵測

之前都是看單張圖片。現在來做**連續偵測**，模擬真正的即時監控系統！

In [ ]:
# === 即時連續偵測（15 秒）===
channel = 'DW 德國之聲'  # 可換頻道
duration = 15             # 偵測秒數
interval = 2              # 每幾秒抓一次

cap = cv2.VideoCapture(STREAMS[channel])
start = time.time()
frame_count = 0
all_detections = []

print(f'開始即時偵測 {channel}（{duration} 秒）...')
print()

while time.time() - start < duration:
    ret, frame = cap.read()
    if not ret:
        print('串流中斷，停止偵測')
        break

    results = model(frame, verbose=False)
    annotated = results[0].plot()
    frame_count += 1

    # 統計
    counts = Counter(model.names[int(b.cls)] for b in results[0].boxes)
    all_detections.append(dict(counts))

    # 在畫面上加入即時資訊
    elapsed = time.time() - start
    info = f'Frame #{frame_count} | {elapsed:.1f}s | {len(results[0].boxes)} objects'
    cv2.putText(annotated, info, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # 顯示（會自動覆蓋上一幀）
    clear_output(wait=True)
    show_frame(annotated, title=f'{channel} - {info}')

    time.sleep(interval)

cap.release()
print(f'\n偵測結束！共處理 {frame_count} 幀')

In [ ]:
# === 偵測統計視覺化 ===
if all_detections:
    # 統計所有幀中出現的物件
    total = Counter()
    for d in all_detections:
        total.update(d)

    if total:
        labels, values = zip(*total.most_common(10))
        colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # 長條圖
        ax1.barh(range(len(labels)), values, color=colors)
        ax1.set_yticks(range(len(labels)))
        ax1.set_yticklabels(labels)
        ax1.set_xlabel('Total Count')
        ax1.set_title('Detected Objects (All Frames)')
        ax1.invert_yaxis()

        # 時間序列：每幀的物件總數
        frame_totals = [sum(d.values()) for d in all_detections]
        ax2.plot(range(1, len(frame_totals)+1), frame_totals, 'o-', color='steelblue', linewidth=2)
        ax2.fill_between(range(1, len(frame_totals)+1), frame_totals, alpha=0.2, color='steelblue')
        ax2.set_xlabel('Frame #')
        ax2.set_ylabel('Object Count')
        ax2.set_title('Objects Per Frame Over Time')
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
    else:
        print('這段時間沒有偵測到任何物件')

---
## Step 6: 模型大小比較（選修）

YOLOv8 有不同大小的模型，越大越精確但越慢：

```
模型      參數量    速度     精確度
──────────────────────────────────
yolov8n   3.2M     最快     ★★☆☆
yolov8s   11.2M    快       ★★★☆
yolov8m   25.9M    中等     ★★★★
yolov8l   43.7M    慢       ★★★★
yolov8x   68.2M    最慢     ★★★★★
```

即時應用通常用 nano (n) 或 small (s)。

In [ ]:
# === 比較 nano vs small 模型 ===
model_n = YOLO('yolov8n.pt')  # 已載入
model_s = YOLO('yolov8s.pt')  # 第一次會下載

frame = grab_frame(STREAMS['Al Jazeera 半島電視台'])

# 計時比較
t1 = time.time()
r_n = model_n(frame, verbose=False)
time_n = time.time() - t1

t2 = time.time()
r_s = model_s(frame, verbose=False)
time_s = time.time() - t2

# 視覺比較
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(r_n[0].plot(), cv2.COLOR_BGR2RGB))
axes[0].set_title(f'YOLOv8n (Nano)\n{time_n*1000:.0f}ms | {len(r_n[0].boxes)} objects')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(r_s[0].plot(), cv2.COLOR_BGR2RGB))
axes[1].set_title(f'YOLOv8s (Small)\n{time_s*1000:.0f}ms | {len(r_s[0].boxes)} objects')
axes[1].axis('off')
plt.suptitle('Model Size Comparison', fontsize=14)
plt.tight_layout()
plt.show()

---
## Module 1 完成！

### 你學會了：
- YOLO 物件偵測的基本概念
- 信心度 (Confidence) 的意義
- 6 種不同場景的 AI 辨識
- 即時連續偵測
- 模型大小的取捨

### 接下來：
→ 開啟 **02_applications.ipynb**，用 AI 做人數計算、文字辨識等實際應用！